In [20]:
import numpy as np
!pip install numpy pandas matplotlib scipy


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Load all NCU data

In [24]:
import pandas as pd

metrics_df_raw: pd.DataFrame = pd.read_csv('metrics/metrics_all_ncu.csv')
metrics_df_raw

,GpuName,SeqLen,HeadDim,Run,Range,Action,dram__cycles_active.avg,gpu__time_duration.sum,l1tex__cycles_active.avg,lts__cycles_active.avg,...,smsp__pcsamp_warps_issue_stalled_short_scoreboard,smsp__warps_issue_stalled_long_scoreboard.avg,smsp__warps_issue_stalled_math_pipe_throttle.avg,smsp__warps_issue_stalled_mio_throttle.avg,smsp__warps_issue_stalled_not_selected.avg,smsp__warps_issue_stalled_short_scoreboard.avg,smsp__warps_issue_stalled_wait.avg,sm__pipe_aluheavy_cycles_active.avg,sm__pipe_fmaheavy_cycles_active.avg,sm__pipe_fmalite_cycles_active.avg
0,a100-sxm4-40gb,256,64,0,0,unrolled_elementwise_kernel,50.7,7456.0,1.601593e+03,1.757700e+03,...,1.0,6.297593e+02,0.000000e+00,0.00000,0.000000e+00,29.888889,2.408889e+02,NaN,NaN,NaN
1,a100-sxm4-40gb,256,64,0,0,unrolled_elementwise_kernel,50.7,7424.0,1.583741e+03,1.842463e+03,...,1.0,6.119074e+02,0.000000e+00,0.00000,0.000000e+00,29.824074,2.408889e+02,NaN,NaN,NaN
2,a100-sxm4-40gb,256,64,0,0,unrolled_elementwise_kernel,50.7,7392.0,1.582528e+03,1.733312e+03,...,3.0,6.324398e+02,0.000000e+00,0.00000,0.000000e+00,29.761574,2.408889e+02,NaN,NaN,NaN
3,a100-sxm4-40gb,256,64,0,0,vectorized_elementwise_kernel,54.9,3616.0,2.510556e+02,9.283125e+02,...,0.0,1.009421e+02,0.000000e+00,0.00000,0.000000e+00,1.629630,1.111111e+01,NaN,NaN,NaN
4,a100-sxm4-40gb,256,64,0,0,vectorized_elementwise_kernel,54.9,3616.0,2.438796e+02,1.060450e+03,...,0.0,9.570602e+01,0.000000e+00,0.00000,0.000000e+00,1.629630,1.111111e+01,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2566,t4,8192,128,2,0,volta_sgemm_128x64_nn,9625981.5,5933664.0,3.467783e+06,5.058044e+06,...,544.0,7.826174e+04,3.940338e+06,338280.45625,3.955725e+06,51981.656250,1.556349e+06,NaN,NaN,NaN
2567,t4,8192,128,2,0,vectorized_elementwise_kernel,92463.0,24160.0,1.223858e+04,1.769500e+04,...,6.0,5.292419e+04,1.435188e+02,210.20625,2.052250e+02,1051.981250,3.356756e+03,NaN,NaN,NaN
2568,t4,8192,128,0,0,fmha_cutlassF_f16_aligned_32x128_rf_sm75,411159.0,5579392.0,2.897558e+06,3.540478e+06,...,18328.0,1.083766e+06,1.906061e+05,389439.61250,1.069107e+05,915427.650000,1.209729e+06,NaN,NaN,NaN
2569,t4,8192,128,1,0,fmha_cutlassF_f16_aligned_32x128_rf_sm75,412807.0,5591008.0,2.903576e+06,3.496220e+06,...,18373.0,1.072384e+06,1.908920e+05,391341.08125,1.065178e+05,917177.125000,1.210074e+06,NaN,NaN,NaN


In [27]:
# convert stall cols to proportions instead of raw sample counts
stall_cols = [c for c in metrics_df_raw.columns if c.startswith('smsp__pcsamp_warps_issue_stalled_')]
metrics_df = metrics_df_raw.copy()
total_samples = metrics_df_raw[stall_cols].sum(axis=1)
metrics_df[stall_cols] = metrics_df_raw[stall_cols].div(total_samples, axis=0)
metrics_df

,GpuName,SeqLen,HeadDim,Run,Range,Action,dram__cycles_active.avg,gpu__time_duration.sum,l1tex__cycles_active.avg,lts__cycles_active.avg,...,smsp__pcsamp_warps_issue_stalled_short_scoreboard,smsp__warps_issue_stalled_long_scoreboard.avg,smsp__warps_issue_stalled_math_pipe_throttle.avg,smsp__warps_issue_stalled_mio_throttle.avg,smsp__warps_issue_stalled_not_selected.avg,smsp__warps_issue_stalled_short_scoreboard.avg,smsp__warps_issue_stalled_wait.avg,sm__pipe_aluheavy_cycles_active.avg,sm__pipe_fmaheavy_cycles_active.avg,sm__pipe_fmalite_cycles_active.avg
0,a100-sxm4-40gb,256,64,0,0,unrolled_elementwise_kernel,50.7,7456.0,1.601593e+03,1.757700e+03,...,0.020833,6.297593e+02,0.000000e+00,0.00000,0.000000e+00,29.888889,2.408889e+02,NaN,NaN,NaN
1,a100-sxm4-40gb,256,64,0,0,unrolled_elementwise_kernel,50.7,7424.0,1.583741e+03,1.842463e+03,...,0.025000,6.119074e+02,0.000000e+00,0.00000,0.000000e+00,29.824074,2.408889e+02,NaN,NaN,NaN
2,a100-sxm4-40gb,256,64,0,0,unrolled_elementwise_kernel,50.7,7392.0,1.582528e+03,1.733312e+03,...,0.096774,6.324398e+02,0.000000e+00,0.00000,0.000000e+00,29.761574,2.408889e+02,NaN,NaN,NaN
3,a100-sxm4-40gb,256,64,0,0,vectorized_elementwise_kernel,54.9,3616.0,2.510556e+02,9.283125e+02,...,NaN,1.009421e+02,0.000000e+00,0.00000,0.000000e+00,1.629630,1.111111e+01,NaN,NaN,NaN
4,a100-sxm4-40gb,256,64,0,0,vectorized_elementwise_kernel,54.9,3616.0,2.438796e+02,1.060450e+03,...,0.000000,9.570602e+01,0.000000e+00,0.00000,0.000000e+00,1.629630,1.111111e+01,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2566,t4,8192,128,2,0,volta_sgemm_128x64_nn,9625981.5,5933664.0,3.467783e+06,5.058044e+06,...,0.006699,7.826174e+04,3.940338e+06,338280.45625,3.955725e+06,51981.656250,1.556349e+06,NaN,NaN,NaN
2567,t4,8192,128,2,0,vectorized_elementwise_kernel,92463.0,24160.0,1.223858e+04,1.769500e+04,...,0.017391,5.292419e+04,1.435188e+02,210.20625,2.052250e+02,1051.981250,3.356756e+03,NaN,NaN,NaN
2568,t4,8192,128,0,0,fmha_cutlassF_f16_aligned_32x128_rf_sm75,411159.0,5579392.0,2.897558e+06,3.540478e+06,...,0.392446,1.083766e+06,1.906061e+05,389439.61250,1.069107e+05,915427.650000,1.209729e+06,NaN,NaN,NaN
2569,t4,8192,128,1,0,fmha_cutlassF_f16_aligned_32x128_rf_sm75,412807.0,5591008.0,2.903576e+06,3.496220e+06,...,0.392619,1.072384e+06,1.908920e+05,391341.08125,1.065178e+05,917177.125000,1.210074e+06,NaN,NaN,NaN


In [28]:
metrics_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2571 entries, 0 to 2570
Data columns (total 45 columns):
 #   Column                                                         Non-Null Count  Dtype  
---  ------                                                         --------------  -----  
 0   GpuName                                                        2571 non-null   str    
 1   SeqLen                                                         2571 non-null   int64  
 2   HeadDim                                                        2571 non-null   int64  
 3   Run                                                            2571 non-null   int64  
 4   Range                                                          2571 non-null   int64  
 5   Action                                                         2571 non-null   str    
 6   dram__cycles_active.avg                                        2571 non-null   float64
 7   gpu__time_duration.sum                                         2571 non

In [29]:
LABEL_COLS = ['GpuName', 'SeqLen', 'HeadDim', 'Run', 'Action', 'Range']

## Compare runs for sanity check

In [30]:
metrics_df_run0 = metrics_df[metrics_df['Run'] == 0].reset_index(drop=True)
metrics_df_run1 = metrics_df[metrics_df['Run'] == 1].reset_index(drop=True)
metrics_df_run2 = metrics_df[metrics_df['Run'] == 2].reset_index(drop=True)
metrics_df_run0

,GpuName,SeqLen,HeadDim,Run,Range,Action,dram__cycles_active.avg,gpu__time_duration.sum,l1tex__cycles_active.avg,lts__cycles_active.avg,...,smsp__pcsamp_warps_issue_stalled_short_scoreboard,smsp__warps_issue_stalled_long_scoreboard.avg,smsp__warps_issue_stalled_math_pipe_throttle.avg,smsp__warps_issue_stalled_mio_throttle.avg,smsp__warps_issue_stalled_not_selected.avg,smsp__warps_issue_stalled_short_scoreboard.avg,smsp__warps_issue_stalled_wait.avg,sm__pipe_aluheavy_cycles_active.avg,sm__pipe_fmaheavy_cycles_active.avg,sm__pipe_fmalite_cycles_active.avg
0,a100-sxm4-40gb,256,64,0,0,unrolled_elementwise_kernel,50.7,7456.0,1.601593e+03,1.757700e+03,...,0.020833,6.297593e+02,0.000000e+00,0.00000,0.000000e+00,29.888889,2.408889e+02,NaN,NaN,NaN
1,a100-sxm4-40gb,256,64,0,0,unrolled_elementwise_kernel,50.7,7424.0,1.583741e+03,1.842463e+03,...,0.025000,6.119074e+02,0.000000e+00,0.00000,0.000000e+00,29.824074,2.408889e+02,NaN,NaN,NaN
2,a100-sxm4-40gb,256,64,0,0,unrolled_elementwise_kernel,50.7,7392.0,1.582528e+03,1.733312e+03,...,0.096774,6.324398e+02,0.000000e+00,0.00000,0.000000e+00,29.761574,2.408889e+02,NaN,NaN,NaN
3,a100-sxm4-40gb,256,64,0,0,vectorized_elementwise_kernel,54.9,3616.0,2.510556e+02,9.283125e+02,...,NaN,1.009421e+02,0.000000e+00,0.00000,0.000000e+00,1.629630,1.111111e+01,NaN,NaN,NaN
4,a100-sxm4-40gb,256,64,0,0,vectorized_elementwise_kernel,54.9,3616.0,2.438796e+02,1.060450e+03,...,0.000000,9.570602e+01,0.000000e+00,0.00000,0.000000e+00,1.629630,1.111111e+01,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
852,t4,8192,128,0,0,elementwise_kernel,9641152.5,2317536.0,1.354124e+06,1.977462e+06,...,0.019228,6.349017e+06,2.223474e+05,739.10000,3.436425e+05,148696.475000,1.318682e+06,NaN,NaN,NaN
853,t4,8192,128,0,0,vectorized_elementwise_kernel,6942755.0,1568448.0,9.150590e+05,1.331819e+06,...,0.017845,6.207038e+06,1.491325e+03,36598.96875,9.233056e+03,106809.768750,2.115388e+05,NaN,NaN,NaN
854,t4,8192,128,0,0,volta_sgemm_128x64_nn,9625974.0,5933856.0,3.467528e+06,5.056996e+06,...,0.007023,7.836398e+04,3.942865e+06,339112.88125,3.958883e+06,51682.506250,1.556348e+06,NaN,NaN,NaN
855,t4,8192,128,0,0,vectorized_elementwise_kernel,92942.0,24640.0,1.231095e+04,1.778759e+04,...,0.014245,5.335439e+04,1.445000e+02,202.98750,2.091875e+02,1055.325000,3.356800e+03,NaN,NaN,NaN


In [31]:
import numpy as np
from scipy.stats import variation

data_cols = [c for c in metrics_df_run0.columns if c not in LABEL_COLS]

# Stack runs along axis 0 → shape (3, n_rows, n_data_cols)
stacked = np.stack([
    metrics_df_run0[data_cols].values,
    metrics_df_run1[data_cols].values,
    metrics_df_run2[data_cols].values
], axis=0)

# CV = std/mean across the 3 runs for each (row, metric) pair
diff_df = metrics_df_run0[LABEL_COLS].copy()
diff_df[data_cols] = variation(stacked, axis=0)

diff_df

,GpuName,SeqLen,HeadDim,Run,Action,Range,dram__cycles_active.avg,gpu__time_duration.sum,l1tex__cycles_active.avg,lts__cycles_active.avg,...,smsp__pcsamp_warps_issue_stalled_short_scoreboard,smsp__warps_issue_stalled_long_scoreboard.avg,smsp__warps_issue_stalled_math_pipe_throttle.avg,smsp__warps_issue_stalled_mio_throttle.avg,smsp__warps_issue_stalled_not_selected.avg,smsp__warps_issue_stalled_short_scoreboard.avg,smsp__warps_issue_stalled_wait.avg,sm__pipe_aluheavy_cycles_active.avg,sm__pipe_fmaheavy_cycles_active.avg,sm__pipe_fmalite_cycles_active.avg
0,a100-sxm4-40gb,256,64,0,unrolled_elementwise_kernel,0,1.401465e-16,0.013363,0.016247,0.017873,...,0.891731,0.005261,NaN,NaN,NaN,0.000774,0.000000e+00,NaN,NaN,NaN
1,a100-sxm4-40gb,256,64,0,unrolled_elementwise_kernel,0,1.401465e-16,0.008175,0.002018,0.013518,...,0.868519,0.012819,NaN,NaN,NaN,0.000253,0.000000e+00,NaN,NaN,NaN
2,a100-sxm4-40gb,256,64,0,unrolled_elementwise_kernel,0,1.401465e-16,0.010957,0.006833,0.008030,...,0.820874,0.007411,NaN,NaN,NaN,0.001463,0.000000e+00,NaN,NaN,NaN
3,a100-sxm4-40gb,256,64,0,vectorized_elementwise_kernel,0,0.000000e+00,0.004184,0.001615,0.006719,...,NaN,0.032590,NaN,NaN,NaN,0.000000,1.598721e-16,NaN,NaN,NaN
4,a100-sxm4-40gb,256,64,0,vectorized_elementwise_kernel,0,0.000000e+00,0.000000,0.002144,0.004748,...,NaN,0.049561,NaN,NaN,NaN,0.000000,1.598721e-16,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
852,t4,8192,128,0,elementwise_kernel,0,3.544249e-05,0.000265,0.000274,0.000268,...,0.018765,0.000340,0.002920,0.026399,0.001377,0.000509,3.749351e-05,NaN,NaN,NaN
853,t4,8192,128,0,vectorized_elementwise_kernel,0,2.845461e-04,0.000451,0.000507,0.000492,...,0.047431,0.002274,0.001035,0.004009,0.004761,0.004666,9.732628e-05,NaN,NaN,NaN
854,t4,8192,128,0,volta_sgemm_128x64_nn,0,1.167204e-05,0.000140,0.000036,0.000117,...,0.031511,0.000625,0.000304,0.003046,0.000348,0.002941,2.091027e-06,NaN,NaN,NaN
855,t4,8192,128,0,vectorized_elementwise_kernel,0,2.977597e-03,0.008035,0.002820,0.002628,...,0.193206,0.004602,0.005474,0.016373,0.012894,0.002427,9.486217e-04,NaN,NaN,NaN


In [32]:
threshold = 0.03
data_cols = [c for c in diff_df.columns if c not in LABEL_COLS]

flagged = (
    diff_df[data_cols]
    .stack()
    .reset_index()
    .rename(columns={'level_1': 'metric', 0: 'cv'})
    .query('cv >= @threshold')
)

label_vals = diff_df.loc[flagged['level_0'], LABEL_COLS].reset_index(drop=True)
flagged_df = pd.concat([label_vals, flagged[['metric', 'cv']].reset_index(drop=True)], axis=1)
flagged_df

,GpuName,SeqLen,HeadDim,Run,Action,Range,metric,cv
0,a100-sxm4-40gb,256,64,0,unrolled_elementwise_kernel,0,smsp__pcsamp_warps_issue_stalled_short_scoreboard,0.891731
1,a100-sxm4-40gb,256,64,0,unrolled_elementwise_kernel,0,smsp__pcsamp_warps_issue_stalled_short_scoreboard,0.868519
2,a100-sxm4-40gb,256,64,0,unrolled_elementwise_kernel,0,smsp__pcsamp_warps_issue_stalled_long_scoreboard,0.041507
3,a100-sxm4-40gb,256,64,0,unrolled_elementwise_kernel,0,smsp__pcsamp_warps_issue_stalled_short_scoreboard,0.820874
4,a100-sxm4-40gb,256,64,0,vectorized_elementwise_kernel,0,smsp__warps_issue_stalled_long_scoreboard.avg,0.032590
...,...,...,...,...,...,...,...,...
2864,t4,8192,128,0,volta_sgemm_128x64_nn,0,smsp__pcsamp_warps_issue_stalled_lg_throttle,0.108126
2865,t4,8192,128,0,volta_sgemm_128x64_nn,0,smsp__pcsamp_warps_issue_stalled_membar,0.228092
2866,t4,8192,128,0,volta_sgemm_128x64_nn,0,smsp__pcsamp_warps_issue_stalled_short_scoreboard,0.031511
2867,t4,8192,128,0,vectorized_elementwise_kernel,0,smsp__pcsamp_warps_issue_stalled_short_scoreboard,0.193206


## Get Utilization Percentages

In [34]:
sm_cycles_elapsed = metrics_df_run0['sm__cycles_elapsed.avg']

In [35]:
util_df = metrics_df_run0[LABEL_COLS]
util_df['SM_Util'] = metrics_df_run0['sm__cycles_active.avg'] / sm_cycles_elapsed
util_df['SMSP_Util'] = metrics_df_run0['smsp__issue_active.avg.pct_of_peak_sustained_elapsed']
util_df['Tensor_Util'] = metrics_df_run0['sm__pipe_tensor_cycles_active.avg'] / sm_cycles_elapsed
util_df['FMA_Util'] = metrics_df_run0['sm__pipe_fma_cycles_active.avg'] / sm_cycles_elapsed
util_df['FMA_Heavy_Util'] = metrics_df_run0['sm__pipe_fmaheavy_cycles_active.avg'] / sm_cycles_elapsed
util_df['FMA_Lite_Util'] = metrics_df_run0['sm__pipe_fmalite_cycles_active.avg'] / sm_cycles_elapsed
util_df['SFU_Util'] = metrics_df_run0['smsp__inst_executed_pipe_xu.avg.pct_of_peak_sustained_elapsed']
util_df['Shared_Util'] = metrics_df_run0['sm__pipe_shared_cycles_active.avg'] / sm_cycles_elapsed
util_df['ALU_Util'] = metrics_df_run0['sm__pipe_alu_cycles_active.avg'] / sm_cycles_elapsed
util_df['ALU_Heavy_Util'] = metrics_df_run0['sm__pipe_aluheavy_cycles_active.avg'] / sm_cycles_elapsed
# util_df['ALU_Lite_Util'] = metrics_df_run0['sm__pipe_alulite_cycles_active.avg'] / sm_cycles_elapsed
# util_df['FP16_Util'] = metrics_df_run0['sm__pipe_fp16_cycles_active.avg'] / sm_cycles_elapsed
util_df['FP64_Util'] = metrics_df_run0['sm__pipe_fp64_cycles_active.avg'] / sm_cycles_elapsed
util_df['L1_Cache_Util'] = metrics_df_run0['l1tex__cycles_active.avg'] / sm_cycles_elapsed
# util_df['L1_Cache_Util_Ampere+'] = metrics_df_run0['1tex__cycles_active.avg'] / sm_cycles_elapsed
util_df['LTS_Util'] = metrics_df_run0['lts__cycles_active.avg'] / sm_cycles_elapsed
util_df['DRAM_Util'] = metrics_df_run0['dram__cycles_active.avg'] / sm_cycles_elapsed
util_df

,GpuName,SeqLen,HeadDim,Run,Action,Range,SM_Util,SMSP_Util,Tensor_Util,FMA_Util,FMA_Heavy_Util,FMA_Lite_Util,SFU_Util,Shared_Util,ALU_Util,ALU_Heavy_Util,FP64_Util,L1_Cache_Util,LTS_Util,DRAM_Util
0,a100-sxm4-40gb,256,64,0,unrolled_elementwise_kernel,0,0.197860,0.735747,0.000000,0.010542,NaN,NaN,0.000000,0.000000,0.019327,NaN,0.000000,0.197860,0.217145,0.006263
1,a100-sxm4-40gb,256,64,0,unrolled_elementwise_kernel,0,0.196005,0.737065,0.000000,0.010561,NaN,NaN,0.000000,0.000000,0.019362,NaN,0.000000,0.196005,0.228025,0.006275
2,a100-sxm4-40gb,256,64,0,unrolled_elementwise_kernel,0,0.197519,0.743326,0.000000,0.010651,NaN,NaN,0.000000,0.000000,0.019526,NaN,0.000000,0.197519,0.216339,0.006328
3,a100-sxm4-40gb,256,64,0,vectorized_elementwise_kernel,0,0.064764,0.122296,0.000306,0.003975,NaN,NaN,0.000000,0.000306,0.000917,NaN,0.000000,0.064764,0.239475,0.014162
4,a100-sxm4-40gb,256,64,0,vectorized_elementwise_kernel,0,0.062554,0.121597,0.000304,0.003952,NaN,NaN,0.000000,0.000304,0.000912,NaN,0.000000,0.062554,0.271999,0.014082
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
852,t4,8192,128,0,elementwise_kernel,0,0.998809,42.061534,0.000000,0.928122,NaN,NaN,0.000000,0.000000,1.198824,NaN,0.000000,0.998809,1.458585,7.111361
853,t4,8192,128,0,vectorized_elementwise_kernel,0,0.997319,5.359269,0.000000,0.114284,NaN,NaN,11.428390,0.000000,0.085713,NaN,0.000000,0.997319,1.451545,7.566882
854,t4,8192,128,0,volta_sgemm_128x64_nn,0,0.998914,56.452366,0.000000,3.875075,NaN,NaN,0.000000,0.000000,0.188403,NaN,0.000000,0.998914,1.456802,2.773018
855,t4,8192,128,0,vectorized_elementwise_kernel,0,0.855973,5.482743,0.000000,0.113917,NaN,NaN,11.391701,0.000000,0.085438,NaN,0.000000,0.855973,1.236761,6.462204


In [36]:
util_df.to_csv('metrics/util_ncu.csv')